# 02 Modelagem e regras de negócio

Investigar as condições que orientam a definição dos produtos, do calendário, dos valores monetários e dos rankings.

### Leitura das fontes

Disponibiliza vendas e classificação de canais e confere a estrutura dos arquivos.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

LANDING = "/Volumes/workspace/beverage_sales_landing/source_files"
sales_raw = (
    spark.read.option("header", True).option("sep", "\t")
    .option("inferSchema", False).option("encoding", "UTF-16")
    .option("multiLine", True).option("mode", "FAILFAST")
    .option("ignoreLeadingWhiteSpace", False).option("ignoreTrailingWhiteSpace", False)
    .csv(f"{LANDING}/abi_bus_case1_beverage_sales_20210726.csv")
)
channels_raw = (
    spark.read.option("header", True).option("sep", ",")
    .option("inferSchema", False).option("encoding", "UTF-8").option("mode", "FAILFAST")
    .option("ignoreLeadingWhiteSpace", False).option("ignoreTrailingWhiteSpace", False)
    .csv(f"{LANDING}/abi_bus_case1_beverage_channel_group_20210726.csv")
)
expected_sales = [
    "DATE", "CE_BRAND_FLVR", "BRAND_NM", "Btlr_Org_LVL_C_Desc", "CHNL_GROUP",
    "TRADE_CHNL_DESC", "PKG_CAT", "Pkg_Cat_Desc", "TSR_PCKG_NM",
    "$ Volume", "YEAR", "MONTH", "PERIOD",
]
expected_channels = ["TRADE_CHNL_DESC", "TRADE_GROUP_DESC", "TRADE_TYPE_DESC"]
if sales_raw.columns != expected_sales or channels_raw.columns != expected_channels:
    raise ValueError("Cabeçalho inesperado. Conferir fontes e codificações antes de continuar.")

## 1 Precisão dos valores

Mede os dígitos inteiros e as casas decimais presentes na fonte. Verifica se os valores cabem no tipo de análise sem perda de precisão.

In [0]:
amount_text = F.trim(F.col("$ Volume"))
precision_profile = (
    sales_raw.select(amount_text.alias("original_amount"))
    .withColumn("valid_format", F.col("original_amount").rlike(r"^[+-]?\d+(\.\d+)?$"))
    .withColumn("decimal_places", F.length(F.regexp_extract("original_amount", r"\.(\d+)$", 1)))
    .withColumn("effective_decimal_places", F.length(
        F.regexp_replace(F.regexp_extract("original_amount", r"\.(\d+)$", 1), r"0+$", "")
    ))
    .withColumn("integer_digits", F.greatest(F.lit(1), F.length(
        F.regexp_replace(F.regexp_extract("original_amount", r"^[+-]?(\d+)", 1), r"^0+", "")
    )))
)
precision_stats = precision_profile.agg(
    F.count("*").alias("rows"),
    F.count(F.when(~F.coalesce(F.col("valid_format"), F.lit(False)), 1)).alias("invalid"),
    F.max("decimal_places").alias("max_scale"),
    F.max("effective_decimal_places").alias("max_effective_scale"),
    F.max("integer_digits").alias("max_integer_digits"),
).first()
display(precision_profile.groupBy("decimal_places", "effective_decimal_places").count()
        .orderBy("decimal_places", "effective_decimal_places"))
if precision_stats["rows"] == 0 or precision_stats["invalid"] > 0:
    raise ValueError("Base vazia ou valores fora do formato decimal simples. Revisar antes das somas.")
observed_scale = int(precision_stats["max_scale"])
if precision_stats["max_integer_digits"] + observed_scale > 28:
    raise ValueError("Precisão observada não cabe na proposta decimal(28, escala). Revisar o tipo.")
amount_type = f"decimal(28,{observed_scale})"
print(
    f"Casas observadas: {observed_scale}; casas efetivas: {precision_stats['max_effective_scale']}; "
    f"dígitos inteiros máximos: {precision_stats['max_integer_digits']}; tipo exploratório: {amount_type}"
)

sales = sales_raw.select(
    F.col("DATE").alias("source_date"),
    F.try_to_timestamp(F.trim(F.col("DATE")), F.lit("M/d/yyyy")).cast("date").alias("sale_date"),
    F.trim("CE_BRAND_FLVR").alias("brand_code"),
    F.trim("BRAND_NM").alias("brand"),
    F.trim("Btlr_Org_LVL_C_Desc").alias("region"),
    F.trim("PKG_CAT").alias("package_code"),
    F.trim("Pkg_Cat_Desc").alias("package_description"),
    F.trim("TSR_PCKG_NM").alias("package_name"),
    F.upper(F.trim("TRADE_CHNL_DESC")).alias("channel_key"),
    F.expr(f"try_cast(trim(`$ Volume`) AS {amount_type})").alias("amount"),
    F.expr("try_cast(trim(`YEAR`) AS int)").alias("source_year"),
    F.expr("try_cast(trim(`MONTH`) AS int)").alias("source_month"),
    F.expr("try_cast(trim(`PERIOD`) AS int)").alias("source_period"),
).withColumn("year_month", F.date_format("sale_date", "yyyy-MM"))
invalid_typed = sales.filter(
    F.col("sale_date").isNull() | F.col("amount").isNull()
    | F.col("source_year").isNull() | F.col("source_month").isNull()
    | F.col("source_period").isNull()
)
if invalid_typed.limit(1).count():
    display(invalid_typed.limit(20))
    raise ValueError("Há falhas de conversão. Revisar antes de interpretar as análises seguintes.")

decimal_places,effective_decimal_places,count
0,0,4209
1,1,4972
2,2,6970


Casas observadas: 2; casas efetivas: 2; dígitos inteiros máximos: 4; tipo exploratório: decimal(28,2)


## 2 Identificação de produtos e embalagens

Compara chaves candidatas e identifica quantas apresentações cada uma reúne. Chaves com mais de um conjunto de atributos não identificam o produto de forma unívoca.

In [0]:
product_attributes = ["brand_code", "brand", "package_code", "package_description", "package_name"]
products = sales.select(*product_attributes).distinct()
candidates = [
    ("codigo_marca_sabor", ["brand_code"]),
    ("codigo_e_categoria_embalagem", ["brand_code", "package_code"]),
    ("codigo_categoria_e_nome_embalagem", ["brand_code", "package_code", "package_name"]),
]
product_summary = []
for label, keys in candidates:
    grouped = products.groupBy(*keys).agg(F.count("*").alias("attribute_variants"))
    result = grouped.agg(
        F.count("*").alias("keys"),
        F.count(F.when(F.col("attribute_variants") > 1, 1)).alias("conflicts"),
        F.max("attribute_variants").alias("max_variants"),
    ).first()
    product_summary.append((label, ", ".join(keys), result["keys"], result["conflicts"], result["max_variants"]))
    if result["conflicts"]:
        print(f"Chaves ambíguas — {label}")
        display(grouped.filter(F.col("attribute_variants") > 1).orderBy(*keys))
display(spark.createDataFrame(
    product_summary,
    "candidato string, campos string, chaves long, chaves_ambiguas long, max_variantes long",
))
display(products.orderBy(*product_attributes))
print("Atributos ausentes/vazios na proposta de produto:")
display(products.agg(*[
    F.count(F.when(F.col(name).isNull() | (F.col(name) == ""), 1)).alias(name)
    for name in product_attributes
]))

Chaves ambíguas — codigo_marca_sabor


brand_code,attribute_variants
3440,8
3441,7
3554,6


Chaves ambíguas — codigo_e_categoria_embalagem


brand_code,package_code,attribute_variants
3440,N20O,4
3440,N56P,3
3441,N20O,4
3441,N56P,3
3554,N20O,4
3554,N56P,2


candidato,campos,chaves,chaves_ambiguas,max_variantes
codigo_marca_sabor,brand_code,4,3,8
codigo_e_categoria_embalagem,"brand_code, package_code",8,6,4
codigo_categoria_e_nome_embalagem,"brand_code, package_code, package_name",22,0,1


brand_code,brand,package_code,package_description,package_name
3440,LEMON,N128,12Z/355M 8NR,12Z NRP 8P F
3440,LEMON,N20O,20Z/600ML,.591L NRP 24L
3440,LEMON,N20O,20Z/600ML,.591L NRP 24L *
3440,LEMON,N20O,20Z/600ML,20Z NRP 24L
3440,LEMON,N20O,20Z/600ML,20z NRP 24L S
3440,LEMON,N56P,500ML 6P,.5L NRP 6P
3440,LEMON,N56P,500ML 6P,.5L NRP 6P S
3440,LEMON,N56P,500ML 6P,.5L NRP 6P*
3441,RASPBERRY,N20O,20Z/600ML,.591L NRP 24L
3441,RASPBERRY,N20O,20Z/600ML,.591L NRP 24L *


Atributos ausentes/vazios na proposta de produto:


brand_code,brand,package_code,package_description,package_name
0,0,0,0,0


## 3 Relação entre data e período

Confere a correspondência entre datas e combinações de ano, mês e período. Os intervalos ajudam a entender a frequência observada, sem definir por si só o significado comercial de PERIOD.

In [0]:
date_mapping = sales.select(
    "sale_date", "source_year", "source_month", "source_period"
).distinct()
date_window = Window.orderBy("sale_date")
date_intervals = (
    sales.select("sale_date").distinct()
    .withColumn("previous_date", F.lag("sale_date").over(date_window))
    .withColumn("days_since_previous", F.datediff("sale_date", "previous_date"))
)
display(date_mapping.join(date_intervals, ["sale_date"], "left").orderBy(
    "sale_date", "source_year", "source_month", "source_period"
))
period_mapping = date_mapping.groupBy("source_year", "source_month", "source_period").agg(
    F.countDistinct("sale_date").alias("distinct_dates"),
    F.min("sale_date").alias("first_date"), F.max("sale_date").alias("last_date"),
)
display(period_mapping.orderBy("source_year", "source_month", "source_period"))
print("Datas associadas a mais de uma combinação de ano/mês/período:")
display(date_mapping.groupBy("sale_date").count().filter(F.col("count") > 1))
print("Divergências entre data e atributos de calendário:")
display(sales.agg(
    F.count(F.when(F.year("sale_date") != F.col("source_year"), 1)).alias("year_mismatches"),
    F.count(F.when(F.month("sale_date") != F.col("source_month"), 1)).alias("month_mismatches"),
))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sale_date,source_year,source_month,source_period,previous_date,days_since_previous
2006-01-01,2006,1,1,null,null
2006-01-08,2006,1,2,2006-01-01,7
2006-01-15,2006,1,3,2006-01-08,7
2006-01-22,2006,1,4,2006-01-15,7
2006-02-01,2006,2,1,2006-01-22,10
2006-02-08,2006,2,2,2006-02-01,7
2006-02-15,2006,2,3,2006-02-08,7
2006-02-22,2006,2,4,2006-02-15,7
2006-03-01,2006,3,1,2006-02-22,7
2006-03-08,2006,3,2,2006-03-01,7


source_year,source_month,source_period,distinct_dates,first_date,last_date
2006,1,1,1,2006-01-01,2006-01-01
2006,1,2,1,2006-01-08,2006-01-08
2006,1,3,1,2006-01-15,2006-01-15
2006,1,4,1,2006-01-22,2006-01-22
2006,2,1,1,2006-02-01,2006-02-01
2006,2,2,1,2006-02-08,2006-02-08
2006,2,3,1,2006-02-15,2006-02-15
2006,2,4,1,2006-02-22,2006-02-22
2006,3,1,1,2006-03-01,2006-03-01
2006,3,2,1,2006-03-08,2006-03-08


Datas associadas a mais de uma combinação de ano/mês/período:


sale_date,count


Divergências entre data e atributos de calendário:


year_mismatches,month_mismatches
0,0


## 4 Impacto dos valores negativos e zeros

Quantifica os registros por sinal e sua participação nas vendas. Compara o total líquido com a soma apenas dos positivos para evidenciar o impacto de excluir valores negativos.

In [0]:
signed_sales = sales.withColumn(
    "amount_class",
    F.when(F.col("amount") < 0, "NEGATIVE").when(F.col("amount") == 0, "ZERO").otherwise("POSITIVE"),
)
sign_summary = signed_sales.groupBy("amount_class").agg(
    F.count("*").alias("rows"), F.sum("amount").alias("signed_total"),
    F.sum(F.abs("amount")).alias("absolute_total"),
)
grand_totals = sales.agg(
    F.count("*").alias("total_rows"),
    F.sum("amount").alias("net_total"),
    F.sum(F.abs("amount")).alias("total_absolute"),
    F.sum(F.when(F.col("amount") > 0, F.col("amount")).otherwise(F.lit(0))).alias("positive_only_total"),
)
display(sign_summary.crossJoin(grand_totals).select(
    "amount_class", "rows", "signed_total", "absolute_total",
    (F.col("rows") * 100.0 / F.col("total_rows")).alias("row_percentage"),
    F.when(F.col("total_absolute") != 0,
           F.col("absolute_total").cast("double") * 100.0 / F.col("total_absolute").cast("double"))
    .alias("absolute_value_percentage"),
).orderBy("amount_class"))
display(grand_totals.withColumn(
    "increase_if_negatives_removed", F.col("positive_only_total") - F.col("net_total")
))
display(signed_sales.groupBy("year_month", "region", "amount_class")
        .agg(F.count("*").alias("rows"), F.sum("amount").alias("total"))
        .orderBy("year_month", "region", "amount_class"))

amount_class,rows,signed_total,absolute_total,row_percentage,absolute_value_percentage
NEGATIVE,260,-4540.71,4540.71,1.609807442263637,0.10419236218775971
POSITIVE,15764,4353465.87,4353465.87,97.60386353786143,99.89580763781224
ZERO,127,0.00,0.00,0.7863290198749303,0.0


total_rows,net_total,total_absolute,positive_only_total,increase_if_negatives_removed
16151,4348925.16,4358006.58,4353465.87,4540.71


year_month,region,amount_class,rows,total
2006-01,CANADA,NEGATIVE,14,-140.97
2006-01,CANADA,POSITIVE,758,94943.92
2006-01,CANADA,ZERO,8,0.00
2006-01,GREAT LAKES,NEGATIVE,10,-127.56
2006-01,GREAT LAKES,POSITIVE,683,210516.88
2006-01,GREAT LAKES,ZERO,8,0.00
2006-01,MIDWEST,NEGATIVE,5,-80.29
2006-01,MIDWEST,POSITIVE,506,168365.28
2006-01,NORTHEAST,NEGATIVE,7,-21.50
2006-01,NORTHEAST,POSITIVE,673,217979.48


## 5 Cobertura por marca região e mês

Identifica combinações sem registros e diferencia essas ausências de combinações com vendas que somam zero. A grade considera somente marcas, regiões e meses observados.

In [0]:
brands = sales.select("brand").distinct()
regions = sales.select("region").distinct()
months = sales.select("year_month").distinct()
monthly_grid = brands.crossJoin(regions).crossJoin(months)
monthly_actual = sales.groupBy("brand", "region", "year_month").agg(
    F.count("*").alias("row_count"), F.sum("amount").alias("net_sales"),
    F.count(F.when(F.col("amount") == 0, 1)).alias("zero_value_rows"),
)
monthly_coverage = (
    monthly_grid.join(monthly_actual, ["brand", "region", "year_month"], "left")
    .withColumn("coverage_status",
                F.when(F.col("row_count").isNull(), "NO_ROWS")
                .when(F.col("net_sales") == 0, "NET_ZERO_WITH_ROWS")
                .otherwise("HAS_ROWS_NONZERO"))
)
display(monthly_coverage.groupBy("coverage_status").count())
display(monthly_coverage.orderBy("brand", "region", "year_month"))

brand_region_actual = sales.groupBy("brand", "region").agg(
    F.count("*").alias("row_count"), F.sum("amount").alias("net_sales"),
)
print("Marcas/regiões sem registros em todo o período observado:")
display(brands.crossJoin(regions).join(brand_region_actual, ["brand", "region"], "left")
        .filter(F.col("row_count").isNull()).orderBy("region", "brand"))

coverage_status,count
HAS_ROWS_NONZERO,64
NO_ROWS,20


brand,region,year_month,row_count,net_sales,zero_value_rows,coverage_status
GRAPE,CANADA,2006-01,null,null,null,NO_ROWS
GRAPE,CANADA,2006-02,null,null,null,NO_ROWS
GRAPE,CANADA,2006-03,null,null,null,NO_ROWS
GRAPE,GREAT LAKES,2006-01,null,null,null,NO_ROWS
GRAPE,GREAT LAKES,2006-02,null,null,null,NO_ROWS
GRAPE,GREAT LAKES,2006-03,null,null,null,NO_ROWS
GRAPE,MIDWEST,2006-01,null,null,null,NO_ROWS
GRAPE,MIDWEST,2006-02,null,null,null,NO_ROWS
GRAPE,MIDWEST,2006-03,null,null,null,NO_ROWS
GRAPE,NORTHEAST,2006-01,null,null,null,NO_ROWS


Marcas/regiões sem registros em todo o período observado:


brand,region,row_count,net_sales
GRAPE,CANADA,null,null
GRAPE,GREAT LAKES,null,null
GRAPE,MIDWEST,null,null
GRAPE,NORTHEAST,null,null
GRAPE,SOUTHEAST,null,null
GRAPE,WEST,null,null


## 6 Empates nos rankings

Verifica se diferentes políticas de desempate alteram o Top 3 e identifica marcas que compartilham o menor valor por região. Os cálculos são diagnósticos para definir as futuras consultas de negócio.

In [0]:
channels = channels_raw.select(
    F.upper(F.trim("TRADE_CHNL_DESC")).alias("channel_key"),
    F.trim("TRADE_GROUP_DESC").alias("trade_group"),
)
bad_channels = channels.filter(
    F.col("channel_key").isNull() | (F.col("channel_key") == "")
    | F.col("trade_group").isNull() | (F.col("trade_group") == "")
)
if bad_channels.limit(1).count() or channels.groupBy("channel_key").count().filter(
    F.col("count") > 1
).limit(1).count():
    raise ValueError("Classificação de canal ausente ou ambígua. Revisar antes dos rankings.")
if sales.join(channels, ["channel_key"], "left_anti").limit(1).count():
    raise ValueError("Há vendas sem classificação de canal. Revisar antes dos rankings.")

trade_totals = sales.join(channels, ["channel_key"], "left").groupBy(
    "region", "trade_group"
).agg(F.sum("amount").alias("net_sales"))
trade_ranked = (
    trade_totals
    .withColumn("position_with_ties", F.rank().over(
        Window.partitionBy("region").orderBy(F.desc("net_sales"))))
    .withColumn("exact_position", F.row_number().over(
        Window.partitionBy("region").orderBy(F.desc("net_sales"), F.asc("trade_group"))))
)
print("Quantidade de grupos retornados por cada política de Top 3:")
display(trade_ranked.groupBy("region").agg(
    F.count("*").alias("groups_with_rows"),
    F.count(F.when(F.col("exact_position") <= 3, 1)).alias("exact_top3_count"),
    F.count(F.when(F.col("position_with_ties") <= 3, 1)).alias("top3_including_ties_count"),
).orderBy("region"))
print("Empates de totais entre grupos comerciais, inclusive fora do Top 3:")
display(trade_totals.groupBy("region", "net_sales").agg(
    F.count("*").alias("tied_groups"), F.sort_array(F.collect_set("trade_group")).alias("groups"),
).filter(F.col("tied_groups") > 1).orderBy("region", F.desc("net_sales")))

brand_ranked = brand_region_actual.withColumn("lowest_rank", F.rank().over(
    Window.partitionBy("region").orderBy(F.asc("net_sales"))
))
print("Quantidade de marcas empatadas no menor valor de cada região:")
display(brand_ranked.filter(F.col("lowest_rank") == 1).groupBy("region").agg(
    F.count("*").alias("brands_at_minimum"),
    F.min("net_sales").alias("minimum_observed_sales"),
    F.sort_array(F.collect_set("brand")).alias("brands"),
).orderBy("region"))

Quantidade de grupos retornados por cada política de Top 3:


region,groups_with_rows,exact_top3_count,top3_including_ties_count
CANADA,6,3,3
GREAT LAKES,6,3,3
MIDWEST,6,3,3
NORTHEAST,6,3,3
SOUTHEAST,6,3,3
SOUTHWEST,6,3,3
WEST,6,3,3


Empates de totais entre grupos comerciais, inclusive fora do Top 3:


region,net_sales,tied_groups,groups


Quantidade de marcas empatadas no menor valor de cada região:


region,brands_at_minimum,minimum_observed_sales,brands
CANADA,1,74009.61,List(STRAWBERRY)
GREAT LAKES,1,208747.90,List(STRAWBERRY)
MIDWEST,1,149206.78,List(STRAWBERRY)
NORTHEAST,1,195581.49,List(STRAWBERRY)
SOUTHEAST,1,223958.50,List(RASPBERRY)
SOUTHWEST,1,7.50,List(GRAPE)
WEST,1,134511.18,List(STRAWBERRY)


## Leitura dos resultados

Relacionar as evidências às decisões de chave de produto, precisão monetária, tratamento dos sinais, ausências e desempates.

As justificativas e as propostas para o pipeline estão em [README](README.md).